# Clef Employee Knowledge Assistant
## Phase 3 — Structure-Aware Semantic Chunking

This notebook converts the cleaned employee handbook documents into
retrieval-ready chunks for the RAG pipeline.

The chunking strategy is structure-aware:

1. Preserve document boundaries.
2. Preserve Markdown heading hierarchy.
3. Group related paragraphs and lists together.
4. Split only when a semantic section becomes too large.
5. Use overlap only when a large semantic section must be split.
6. Attach document and section metadata to every chunk.

The goal is to create chunks that represent coherent pieces of
employee knowledge rather than arbitrary portions of text.

## Structure
01. Objective
02. Load cleaned documents
03. Inspect document structure
04. Parse Markdown headings
05. Build hierarchical sections
06. Create semantic units
07. Apply size constraints
08. Create chunks
09. Generate chunk metadata
10. Inspect chunks
11. Analyze chunk statistics
12. Detect problematic chunks
13. Save chunks

## Load cleaned documents

In [29]:
!pip install pandas

In [1]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.2.6
Pandas: 2.2.3


In [2]:
from pathlib import Path
import re
import json
import pandas as pd

PROJECT_ROOT = Path(r"E:\Projects\Speech AI\Data")

PROCESSED_DIR = PROJECT_ROOT / "processed"
CLEANED_DIR = PROCESSED_DIR / "cleaned"
METADATA_DIR = PROJECT_ROOT / "metadata"
CHUNKS_DIR = PROJECT_ROOT / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("Cleaned documents:", CLEANED_DIR)
print("Metadata:", METADATA_DIR)
print("Chunks:", CHUNKS_DIR)

Cleaned documents: E:\Projects\Speech AI\Data\processed\cleaned
Metadata: E:\Projects\Speech AI\Data\metadata
Chunks: E:\Projects\Speech AI\Data\chunks


## Inspect heading structure first

Before we write the actual splitter, I strongly recommend doing this.

In [3]:
def extract_headings(text):
    """
    Extract Markdown headings while preserving their level.
    """
    headings = []

    for line in text.splitlines():
        match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)

        if match:
            level = len(match.group(1))
            title = match.group(2).strip()

            headings.append({
                "level": level,
                "title": title
            })

    return headings

In [4]:
cleaned_files = sorted(CLEANED_DIR.rglob("*.md"))

heading_rows = []

for file_path in cleaned_files:

    text = file_path.read_text(encoding="utf-8")

    headings = extract_headings(text)

    for heading in headings:
        heading_rows.append({
            "file": str(
                file_path.relative_to(CLEANED_DIR)
            ).replace("\\", "/"),
            "level": heading["level"],
            "heading": heading["title"]
        })

headings_df = pd.DataFrame(heading_rows)

headings_df.head(30)

,file,level,heading
0,Benefits and Perks/Continuing Education.md,1,Continuing Education
1,Benefits and Perks/Continuing Education.md,2,Learning Budget
2,Benefits and Perks/Continuing Education.md,2,Mentorship
3,Benefits and Perks/Continuing Education.md,2,Speaker Support
4,Benefits and Perks/Healthcare and Disability I...,1,Healthcare and Disability Insurance
5,Benefits and Perks/Holiday List.md,1,Clef Observed Holiday List
6,Benefits and Perks/New Parent Leave.md,1,New Parent Leave
7,Benefits and Perks/Other Protected Absences.md,1,Other Protected Absences
8,Benefits and Perks/Other Protected Absences.md,2,Pregnancy Disability Leave
9,Benefits and Perks/Other Protected Absences.md,2,Bereavement Leave


In [5]:
print("Heading level distribution:")
print(headings_df["level"].value_counts().sort_index())

print("\nMaximum heading level:", headings_df["level"].max())

Heading level distribution:
level
1    29
2    47
3    19
4     9
Name: count, dtype: int64

Maximum heading level: 4


In [6]:
headings_df.sort_values("level", ascending=False)[:40]

,file,level,heading
59,Onboarding Documents/Communication and Transpa...,4,Slack Status
48,Employment Policies/Working Remotely.md,4,Loss of the privilege
46,Employment Policies/Working Remotely.md,4,Regular 1:1s
47,Employment Policies/Working Remotely.md,4,Dedicated Retrospectives
44,Employment Policies/Working Remotely.md,4,Co-working Space Subsidies
43,Employment Policies/Working Remotely.md,4,Plan & Prepare Beforehand
42,Employment Policies/Working Remotely.md,4,Give the team heads up
41,Employment Policies/Working Remotely.md,4,Extended remote work
58,Onboarding Documents/Communication and Transpa...,4,Slack Names
40,Employment Policies/Working Remotely.md,3,Extended Remote Work


## 4. Analyze Section Sizes

Before selecting chunk-size parameters, we measure the size of
the existing semantic sections.

This allows the chunking strategy to be based on the actual corpus
rather than an arbitrary token limit.

In [7]:
def extract_sections(text):
    """
    Extract top-level Markdown sections based on H1/H2/H3/H4 headings.

    Each section contains:
    - heading level
    - heading title
    - content belonging to that heading
    """

    lines = text.splitlines()

    sections = []
    current = None

    for line in lines:

        match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)

        if match:

            if current is not None:
                current["content"] = "\n".join(
                    current["content"]
                ).strip()

                sections.append(current)

            current = {
                "level": len(match.group(1)),
                "heading": match.group(2).strip(),
                "content": []
            }

        else:

            if current is not None:
                current["content"].append(line)

    # Save final section
    if current is not None:
        current["content"] = "\n".join(
            current["content"]
        ).strip()

        sections.append(current)

    return sections

In [8]:
section_rows = []

for file_path in cleaned_files:

    text = file_path.read_text(encoding="utf-8")

    sections = extract_sections(text)

    for section in sections:

        content = section["content"]

        section_rows.append({
            "file": str(
                file_path.relative_to(CLEANED_DIR)
            ).replace("\\", "/"),
            "level": section["level"],
            "heading": section["heading"],
            "characters": len(content),
            "words": len(content.split())
        })

sections_df = pd.DataFrame(section_rows)

sections_df.head(20)

,file,level,heading,characters,words
0,Benefits and Perks/Continuing Education.md,1,Continuing Education,243,43
1,Benefits and Perks/Continuing Education.md,2,Learning Budget,1038,174
2,Benefits and Perks/Continuing Education.md,2,Mentorship,583,97
3,Benefits and Perks/Continuing Education.md,2,Speaker Support,747,127
4,Benefits and Perks/Healthcare and Disability I...,1,Healthcare and Disability Insurance,1205,193
5,Benefits and Perks/Holiday List.md,1,Clef Observed Holiday List,1172,142
6,Benefits and Perks/New Parent Leave.md,1,New Parent Leave,1481,266
7,Benefits and Perks/Other Protected Absences.md,1,Other Protected Absences,0,0
8,Benefits and Perks/Other Protected Absences.md,2,Pregnancy Disability Leave,491,78
9,Benefits and Perks/Other Protected Absences.md,2,Bereavement Leave,451,64


In [9]:
print("\nLargest sections:")

sections_df.sort_values(
    "words",
    ascending=False
)[
    [
        "file",
        "level",
        "heading",
        "words"
    ]
].head(20)


Largest sections:


,file,level,heading,words
74,Onboarding Documents/Product Manifesto.md,1,Product Manifesto,577
35,Employment Policies/Salary and Equity Compensa...,2,Salary,447
22,Employment Policies/Complaint Policy.md,1,Complaint Policy,364
69,Onboarding Documents/Objectives and Key Result...,2,Scoring and Evaluating OKRs,351
12,Benefits and Perks/Referral Bonuses.md,1,Referral Bonuses,336
21,Employment Policies/Code of Conduct in the Com...,1,Code of Conduct,296
38,Employment Policies/Working Remotely.md,2,Approach,291
13,Benefits and Perks/Sabbatical.md,1,Sabbatical,287
6,Benefits and Perks/New Parent Leave.md,1,New Parent Leave,266
79,Onboarding Documents/Welcome to Clef.md,2,"The Clef Team, Hours, and Office",264


In [10]:
sections_df["words"].describe()

count    104.000000
mean     111.740385
std      102.139348
min        0.000000
25%       46.000000
50%       81.500000
75%      150.500000
max      577.000000
Name: words, dtype: float64

## Final chunking strategy

So I recommend:
```
                    CLEANED DOCUMENT
                           │
                           ▼
                    Parse headings
                           │
                           ▼
                   Build hierarchy
                           │
                           ▼
               Identify semantic sections
                           │
             ┌─────────────┴─────────────┐
             │                           │
        ≤ 350 words                  > 350 words
             │                           │
             ▼                           ▼
       Keep intact                Recursive splitting
                                         │
                                  ┌──────┴──────┐
                                  ▼             ▼
                              Paragraphs    Sentences
                                         
```

## Chunk rules

Situation	Action

H1/H2/H3/H4 section ≤350 words	Keep intact

Section >350 words	Split at paragraph boundaries

Paragraph still too large	Split at sentences

Empty heading	No chunk

Adjacent tiny sections	Don't automatically merge yet

Different H2 sections	Never mix them

Different H1 documents	Never mix them

Overlap	Only when a large section is split

## 5. Build the Document Hierarchy

Each chunk will retain its complete heading path.

For example:

Document: Other Protected Absences

Section Path:
Other Protected Absences
→ Pregnancy Disability Leave

This allows the retrieval system to understand the context of a
section even when the section's own text is short.

Empty headings are treated as structural parents and do not
generate independent chunks.

In [11]:
def parse_markdown_structure(text):
    """
    Parse a Markdown document into heading-aware sections.

    Returns a list of sections containing:
    - heading level
    - heading title
    - content
    - section path
    """

    lines = text.splitlines()

    sections = []
    heading_stack = []
    current = None

    def save_current():
        if current is not None:
            content = "\n".join(
                current["content"]
            ).strip()

            sections.append({
                "level": current["level"],
                "heading": current["heading"],
                "content": content,
                "section_path": current["section_path"]
            })

    for line in lines:

        match = re.match(
            r"^(#{1,6})\s+(.+?)\s*$",
            line
        )

        if match:

            # Save previous section
            save_current()

            level = len(match.group(1))
            heading = match.group(2).strip()

            # Remove headings from stack that are
            # at the same or deeper level
            while (
                heading_stack
                and heading_stack[-1]["level"] >= level
            ):
                heading_stack.pop()

            heading_stack.append({
                "level": level,
                "heading": heading
            })

            current = {
                "level": level,
                "heading": heading,
                "content": [],
                "section_path": [
                    item["heading"]
                    for item in heading_stack
                ]
            }

        else:

            if current is not None:
                current["content"].append(line)

    # Save final section
    save_current()

    return sections

In [12]:
employee_privacy = (
    CLEANED_DIR
    / "Employment Policies"
    / "Employee Privacy.md"
)

text = employee_privacy.read_text(
    encoding="utf-8"
)

sections = parse_markdown_structure(text)

for section in sections:

    print(
        f"Level {section['level']} | "
        f"{' → '.join(section['section_path'])}"
    )

    print(
        f"Words: {len(section['content'].split())}"
    )

    print("-" * 80)

Level 1 | Employee Privacy
Words: 0
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → Workspace Privacy
Words: 177
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → Email and Internet Privacy
Words: 62
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → Email Is Not Private
Words: 67
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → Use of the Email System for Personal Email
Words: 57
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → All Conduct Rules Apply to Email
Words: 53
--------------------------------------------------------------------------------
Level 2 | Employee Privacy → Professional Tone and Content
Words: 82
-----------------------------------------------------------------------

Then implement the 350-word rule

We'll use a word-based threshold for the first implementation, because your corpus analysis was also word-based and it keeps the notebook easy to understand.

Later, when we choose the embedding model, we'll add a proper tokenizer-based measurement.

In [13]:
TARGET_WORDS = 300
MAX_WORDS = 350

print("Target chunk size:", TARGET_WORDS, "words")
print("Maximum chunk size:", MAX_WORDS, "words")

Target chunk size: 300 words
Maximum chunk size: 350 words


In [14]:
employee_privacy = (
    CLEANED_DIR
    / "Onboarding Documents"
    / "Product Manifesto.md"
)

text = employee_privacy.read_text(
    encoding="utf-8"
)

sections = parse_markdown_structure(text)

for section in sections:

    print(
        f"Level {section['level']} | "
        f"{' → '.join(section['section_path'])}"
    )

    print(
        f"Words: {len(section['content'].split())}"
    )

    print("-" * 80)

Level 1 | Product Manifesto
Words: 577
--------------------------------------------------------------------------------


## 6. Chunking Configuration

The chunker prioritizes semantic boundaries over fixed chunk sizes.

Rules:
- Keep a section intact when it is reasonably small.
- Split large sections at paragraph/list boundaries.
- Keep Markdown lists together.
- Split oversized paragraphs only at sentence boundaries.
- Never cross document or heading boundaries.
- Preserve the complete heading path as context.

In [15]:
TARGET_WORDS = 300
MAX_WORDS = 350
MIN_WORDS = 50

print(f"Target chunk size : {TARGET_WORDS} words")
print(f"Maximum chunk size: {MAX_WORDS} words")
print(f"Minimum chunk size: {MIN_WORDS} words")

Target chunk size : 300 words
Maximum chunk size: 350 words
Minimum chunk size: 50 words


7. Create semantic units

The first important step is converting the content into paragraph/list units.

We don't want to split lists item-by-item.

In [16]:
def extract_semantic_units(content):
    """
    Convert Markdown content into semantic units.

    Rules:
    - Normal paragraphs are separate units.
    - Consecutive Markdown list items are kept together,
      even when blank lines exist between them.
    - A paragraph immediately before a list is merged
      with the complete list.
    - Code blocks remain intact.
    """

    lines = content.splitlines()

    units = []

    current_paragraph = []
    current_list = []

    in_code_block = False
    code_block = []

    def save_paragraph():
        nonlocal current_paragraph

        if current_paragraph:
            text = "\n".join(current_paragraph).strip()

            if text:
                units.append({
                    "text": text,
                    "type": "paragraph"
                })

            current_paragraph = []

    def save_list():
        nonlocal current_list

        if current_list:
            text = "\n".join(current_list).strip()

            if text:
                units.append({
                    "text": text,
                    "type": "list"
                })

            current_list = []

    def save_code():
        nonlocal code_block

        if code_block:
            text = "\n".join(code_block).strip()

            if text:
                units.append({
                    "text": text,
                    "type": "code"
                })

            code_block = []

    for line in lines:

        stripped = line.strip()

        # ==================================================
        # CODE BLOCK
        # ==================================================

        if stripped.startswith("```"):

            if not in_code_block:

                # Save anything before the code block
                save_paragraph()
                save_list()

                in_code_block = True
                code_block.append(line)

            else:

                code_block.append(line)
                save_code()

                in_code_block = False

            continue

        if in_code_block:

            code_block.append(line)
            continue

        # ==================================================
        # LIST ITEM
        # ==================================================

        is_list_item = bool(
            re.match(
                r"^\s*(?:[-*+]|\d+\.)\s+",
                line
            )
        )

        if is_list_item:

            # If we encounter a list after a paragraph,
            # save the paragraph first.
            save_paragraph()

            current_list.append(line)

            continue

        # ==================================================
        # BLANK LINE
        # ==================================================

        if not stripped:

            # IMPORTANT:
            # Do NOT save a list here.
            #
            # Markdown lists can contain blank lines
            # between items.
            if current_list:
                current_list.append(line)

            else:
                save_paragraph()

            continue

        # ==================================================
        # CONTINUATION OF LIST
        # ==================================================

        if current_list:

            # Indented continuation of a list item
            if line.startswith((" ", "\t")):

                current_list.append(line)
                continue

            # Otherwise list has ended
            save_list()

        # ==================================================
        # NORMAL PARAGRAPH
        # ==================================================

        current_paragraph.append(line)

    # ======================================================
    # SAVE REMAINING CONTENT
    # ======================================================

    save_paragraph()
    save_list()
    save_code()

    # ======================================================
    # MERGE PARAGRAPH + FOLLOWING LIST
    # ======================================================

    merged_units = []

    i = 0

    while i < len(units):

        current = units[i]

        if (
            current["type"] == "paragraph"
            and i + 1 < len(units)
            and units[i + 1]["type"] == "list"
        ):

            merged_text = (
                current["text"]
                + "\n\n"
                + units[i + 1]["text"]
            )

            merged_units.append({
                "text": merged_text,
                "type": "atomic"
            })

            i += 2

        else:

            merged_units.append(current)

            i += 1

    return merged_units

8. Test the semantic-unit parser

Let's first test it on Product Manifesto.

## 9. Verify the list specifically


In [17]:
for section in sections:

    units = extract_semantic_units(
        section["content"]
    )

    for i, unit in enumerate(units):

        if "Guilt" in unit and "Anxiety" in unit:

            print("Found combined emotional-response list:")
            print(unit)

## 10. Split oversized paragraphs when necessary

Most of your documents won't need this.

But we need a fallback for a paragraph that's, say: 500 words.
We should split it by sentences rather than characters.

In [18]:
def split_into_sentences(text):
    """
    Lightweight sentence splitter.

    This intentionally avoids requiring an NLP library
    at this stage of preprocessing.
    """

    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        text.strip()
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

## 11. Test sentence splitting

In [19]:
test_text = """
Passwords take time and are fundamentally insecure, but most people
log in pretty quickly and don't know enough to understand the security
minutiae. What we found to be more important are the things people
feel when logging in. These emotional responses influence how people
think about identity.
"""

sentences = split_into_sentences(test_text)

for i, sentence in enumerate(sentences, 1):

    print(f"{i}. {sentence}")

1. Passwords take time and are fundamentally insecure, but most people
log in pretty quickly and don't know enough to understand the security
minutiae.
2. What we found to be more important are the things people
feel when logging in.
3. These emotional responses influence how people
think about identity.


## 12. Build the paragraph grouping algorithm

Now comes the main part.

The algorithm tries to create chunks close to TARGET_WORDS, but never exceeds MAX_WORDS unless a single semantic unit itself is larger than the maximum.

In [20]:
def is_atomic_unit(unit):
    """
    A unit is atomic when it is:
    - a complete list
    - a paragraph + list combination
    - a code block
    """

    return unit["type"] in {
        "list",
        "atomic",
        "code"
    }

In [21]:
def group_semantic_units(
    units,
    target_words=TARGET_WORDS,
    max_words=MAX_WORDS
):
    """
    Group semantic units into retrieval-friendly chunks.

    Atomic units are never split.
    """

    chunks = []

    current_units = []
    current_words = 0

    for unit in units:

        text = unit["text"]
        unit_words = len(text.split())

        # ==================================================
        # ATOMIC UNIT
        # ==================================================

        if is_atomic_unit(unit):

            # If there is already content in the current
            # chunk, save it before adding the atomic unit
            # when the combined size would be too large.
            if (
                current_units
                and current_words + unit_words > max_words
            ):

                chunks.append(
                    "\n\n".join(current_units)
                )

                current_units = []
                current_words = 0

            # Add the ENTIRE atomic unit
            current_units.append(text)
            current_words += unit_words

            continue

        # ==================================================
        # NORMAL UNIT
        # ==================================================

        if not current_units:

            current_units.append(text)
            current_words = unit_words

            continue

        if current_words + unit_words <= max_words:

            current_units.append(text)
            current_words += unit_words

        else:

            chunks.append(
                "\n\n".join(current_units)
            )

            current_units = [text]
            current_words = unit_words

    # ======================================================
    # FINAL CHUNK
    # ======================================================

    if current_units:

        chunks.append(
            "\n\n".join(current_units)
        )

    return chunks

Important correction to the previous rule

We're using:

TARGET = 300
MAX = 350

but the function above primarily uses MAX.

That's intentional for now.

We don't want to split a coherent section merely because we've reached 300.

For example:

Paragraph 1 = 180 words
Paragraph 2 = 140 words

Total:

320 words

We keep both together because it's still below our maximum.

Semantic coherence wins.

## 14. Handle a single oversized unit

Now suppose:

Paragraph = 500 words

The previous function would create a 500-word chunk.

That's undesirable.

So let's add a recursive fallback.

In [22]:
def split_large_unit(
    text,
    max_words=MAX_WORDS
):
    """
    Split an oversized non-atomic unit at sentence boundaries.

    Atomic units are NOT split because they may contain
    lists or other information that must remain together.
    """

    # Never split an atomic unit.
    if is_atomic_unit(text):
        return [text]

    word_count = len(text.split())

    if word_count <= max_words:
        return [text]

    sentences = split_into_sentences(text)

    chunks = []

    current = []
    current_words = 0

    for sentence in sentences:

        sentence_words = len(sentence.split())

        if (
            current
            and current_words + sentence_words > max_words
        ):

            chunks.append(
                " ".join(current)
            )

            current = []
            current_words = 0

        current.append(sentence)
        current_words += sentence_words

    if current:

        chunks.append(
            " ".join(current)
        )

    return chunks

## 15. Create the final semantic chunker

Now combine everything.

In [23]:
def create_content_chunks(
    content,
    target_words=TARGET_WORDS,
    max_words=MAX_WORDS
):
    """
    Create structure-aware semantic chunks.
    """

    units = extract_semantic_units(content)

    processed_units = []

    for unit in units:

        text = unit["text"]
        word_count = len(text.split())

        # -----------------------------------------------
        # Atomic units are NEVER split
        # -----------------------------------------------

        if is_atomic_unit(unit):

            processed_units.append(unit)

        # -----------------------------------------------
        # Large normal paragraph
        # -----------------------------------------------

        elif word_count > max_words:

            split_parts = split_large_unit(
                text,
                max_words=max_words
            )

            for part in split_parts:

                processed_units.append({
                    "text": part,
                    "type": "paragraph"
                })

        else:

            processed_units.append(unit)

    return group_semantic_units(
        processed_units,
        target_words=target_words,
        max_words=max_words
    )

## 16. Test it on Product Manifesto

In [24]:
product_sections = parse_markdown_structure(text)

for section in product_sections:

    units = extract_semantic_units(
        section["content"]
    )

    print(
        f"\nSECTION: "
        f"{' → '.join(section['section_path'])}"
    )

    for i, unit in enumerate(units, 1):

        print(
            f"\nUNIT {i}"
            f" | TYPE: {unit['type']}"
            f" | WORDS: {len(unit['text'].split())}"
        )

        print(unit["text"])


SECTION: Product Manifesto

UNIT 1 | TYPE: atomic | WORDS: 92
Working on Clef, it is tempting to optimize for some of the metrics which have been important to other companies like ours. We can use these metrics to compare ourselves against passwords and traditional two-factor authentication, and eke out incremental benefits for users or sites who decide to use Clef. Those metrics might include:

* The level of security we are able to offer customers

* The speed of each login

* The ratio of time logged in with Clef to time managing Clef

* The number of taps between each login

UNIT 2 | TYPE: paragraph | WORDS: 78
For the first two years of Clef’s existence, we often found ourselves following our guts in the wrong direction on each of these metrics but were unable to articulate why. We looked at other solutions that out-performed us in each of these categories and worried about whether we were overlooking optimizations that would make Clef a better product. We kept making decisions t

In [25]:
for section in product_sections:

    chunks = create_content_chunks(
        section["content"]
    )

    print(
        f"\nSECTION: "
        f"{' → '.join(section['section_path'])}"
    )

    print(
        f"Generated chunks: {len(chunks)}"
    )

    for i, chunk in enumerate(chunks, 1):

        print(
            f"\n--- Chunk {i} "
            f"({len(chunk.split())} words) ---"
        )

        print(chunk)


SECTION: Product Manifesto
Generated chunks: 2

--- Chunk 1 (253 words) ---
Working on Clef, it is tempting to optimize for some of the metrics which have been important to other companies like ours. We can use these metrics to compare ourselves against passwords and traditional two-factor authentication, and eke out incremental benefits for users or sites who decide to use Clef. Those metrics might include:

* The level of security we are able to offer customers

* The speed of each login

* The ratio of time logged in with Clef to time managing Clef

* The number of taps between each login

For the first two years of Clef’s existence, we often found ourselves following our guts in the wrong direction on each of these metrics but were unable to articulate why. We looked at other solutions that out-performed us in each of these categories and worried about whether we were overlooking optimizations that would make Clef a better product. We kept making decisions that felt right, but wer

## 21. Define the processed dataset path

In [26]:
from pathlib import Path
import json
import pandas as pd

CLEANED_DIR = Path("processed/cleaned")

print("Processed directory:", CLEANED_DIR.resolve())

Processed directory: E:\Projects\Speech AI\Data\processed\cleaned


In [27]:
markdown_files = sorted(
    CLEANED_DIR.rglob("*.md")
)

print("Markdown files found:", len(markdown_files))

for file in markdown_files:
    print(file.relative_to(CLEANED_DIR))

Markdown files found: 30
Benefits and Perks\Continuing Education.md
Benefits and Perks\Healthcare and Disability Insurance.md
Benefits and Perks\Holiday List.md
Benefits and Perks\New Parent Leave.md
Benefits and Perks\Other Protected Absences.md
Benefits and Perks\Referral Bonuses.md
Benefits and Perks\Sabbatical.md
Benefits and Perks\Vacation and Sick Leave.md
Clef Values.md
Employment Policies\At-Will Employment.md
Employment Policies\Code of Conduct in the Community.md
Employment Policies\Complaint Policy.md
Employment Policies\Drug and Alcohol Policy.md
Employment Policies\Employee Privacy.md
Employment Policies\Equal Opportunity Employment.md
Employment Policies\Salary and Equity Compensation.md
Employment Policies\Working Remotely.md
Hiring Documents\Handbook Introduction.md
Mission Statement.md
Onboarding Documents\Communication and Transparency.md
Onboarding Documents\Direct Reports.md
Onboarding Documents\Objectives and Key Results.md
Onboarding Documents\One on Ones.md
Onboa

## 23. Create the output directory

In [28]:
CHUNK_DIR = Path("chunked_data")

CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Chunk output directory:", CHUNK_DIR.resolve())

Chunk output directory: E:\Projects\Speech AI\Data\chunked_data


24. Helper function for document category

We want metadata telling us where each document came from.

In [29]:
def get_document_category(file_path):
    """
    Determine the document category from its folder.
    """

    relative_path = file_path.relative_to(CLEANED_DIR)

    parts = relative_path.parts

    if len(parts) > 1:
        return parts[0]

    return "General"

In [30]:
for file in markdown_files[:10]:

    print(
        file.name,
        "→",
        get_document_category(file)
    )

Continuing Education.md → Benefits and Perks
Healthcare and Disability Insurance.md → Benefits and Perks
Holiday List.md → Benefits and Perks
New Parent Leave.md → Benefits and Perks
Other Protected Absences.md → Benefits and Perks
Referral Bonuses.md → Benefits and Perks
Sabbatical.md → Benefits and Perks
Vacation and Sick Leave.md → Benefits and Perks
Clef Values.md → General
At-Will Employment.md → Employment Policies


## 25. Read the documents

In [31]:
documents = []

for file_path in markdown_files:

    content = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "file_path": str(
            file_path.relative_to(CLEANED_DIR)
        ),
        "file_name": file_path.name,
        "category": get_document_category(file_path),
        "content": content
    })

print("Documents loaded:", len(documents))

Documents loaded: 30


In [32]:
documents[:2]

[{'file_path': 'Benefits and Perks\\Continuing Education.md',
  'file_name': 'Continuing Education.md',
  'category': 'Benefits and Perks',
  'content': '# Continuing Education\n\nOne of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture.\n\n## Learning Budget\n\nEvery employee has a company budget to support any learning activity that they want to pursue related to the work they do at Clef. This doesn’t need to be a class explicitly linked to their current role, but it should help them improve a skill that will be useful for them at Clef. Each employee has an annual budget of $4,000 which can be spent towards program fees/tuition, tickets, flights, and hotels for industry conferences, classes, mentorship programs, books, programs, videos, or other places that they feel will provide valuable learning exp

## 26. Generate semantic chunks for every document

This is the main processing cell.

In [33]:
all_chunks = []

for doc in documents:

    print(
        f"Processing: {doc['file_path']}"
    )

    sections = parse_markdown_structure(
        doc["content"]
    )

    document_chunk_count = 0

    for section in sections:

        section_path = section["section_path"]
        section_content = section["content"]

        if not section_content.strip():
            continue

        chunks = create_content_chunks(
            section_content,
            target_words=TARGET_WORDS,
            max_words=MAX_WORDS
        )

        for chunk_index, chunk in enumerate(
            chunks,
            start=1
        ):

            chunk_record = {
                "chunk_id": (
                    f"{doc['file_name']}"
                    f"__"
                    f"{chunk_index}"
                ),

                "document": doc["file_name"],

                "file_path": doc["file_path"],

                "category": doc["category"],

                "section_path": section_path,

                "section_title": (
                    section_path[-1]
                    if section_path
                    else doc["file_name"]
                ),

                "chunk_index": chunk_index,

                "word_count": len(
                    chunk.split()
                ),

                "character_count": len(
                    chunk
                ),

                "content": chunk
            }

            all_chunks.append(
                chunk_record
            )

            document_chunk_count += 1

    print(
        f"  → {document_chunk_count} chunks"
    )

print("\nTotal chunks:", len(all_chunks))

Processing: Benefits and Perks\Continuing Education.md
  → 4 chunks
Processing: Benefits and Perks\Healthcare and Disability Insurance.md
  → 1 chunks
Processing: Benefits and Perks\Holiday List.md
  → 1 chunks
Processing: Benefits and Perks\New Parent Leave.md
  → 1 chunks
Processing: Benefits and Perks\Other Protected Absences.md
  → 4 chunks
Processing: Benefits and Perks\Referral Bonuses.md
  → 1 chunks
Processing: Benefits and Perks\Sabbatical.md
  → 1 chunks
Processing: Benefits and Perks\Vacation and Sick Leave.md
  → 1 chunks
Processing: Clef Values.md
  → 4 chunks
Processing: Employment Policies\At-Will Employment.md
  → 1 chunks
Processing: Employment Policies\Code of Conduct in the Community.md
  → 1 chunks
Processing: Employment Policies\Complaint Policy.md
  → 2 chunks
Processing: Employment Policies\Drug and Alcohol Policy.md
  → 1 chunks
Processing: Employment Policies\Employee Privacy.md
  → 8 chunks
Processing: Employment Policies\Equal Opportunity Employment.md
  → 1 

## 27. Convert chunks to DataFrame

In [34]:
chunks_df = pd.DataFrame(
    all_chunks
)

print(
    "Total chunks:",
    len(chunks_df)
)

print(
    "Columns:",
    list(chunks_df.columns)
)

Total chunks: 96
Columns: ['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content']


In [35]:
chunks_df.head()

,chunk_id,document,file_path,category,section_path,section_title,chunk_index,word_count,character_count,content
0,Continuing Education.md__1,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,[Continuing Education],Continuing Education,1,43,243,One of Clef’s core values is “Be better today ...
1,Continuing Education.md__1,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Learning Budget]",Learning Budget,1,174,1038,Every employee has a company budget to support...
2,Continuing Education.md__1,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Mentorship]",Mentorship,1,97,583,Clef understands the value of mentorship. We ...
3,Continuing Education.md__1,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Speaker Support]",Speaker Support,1,127,747,It makes Clef look great when our employees sp...
4,Healthcare and Disability Insurance.md__1,Healthcare and Disability Insurance.md,Benefits and Perks\Healthcare and Disability I...,Benefits and Perks,[Healthcare and Disability Insurance],Healthcare and Disability Insurance,1,193,1205,Clef’s priorities with benefits are wellness a...


In [36]:
chunks_df["word_count"].describe()

count     96.000000
mean     121.052083
std       83.352816
min       24.000000
25%       56.000000
50%       97.500000
75%      160.000000
max      336.000000
Name: word_count, dtype: float64

## 29. Check chunks by document

In [37]:
chunks_per_document = (
    chunks_df
    .groupby("document")
    .size()
    .sort_values(
        ascending=False
    )
)

chunks_per_document

document
Policy Changes.md                         12
Communication and Transparency.md         10
Working Remotely.md                       10
Employee Privacy.md                        8
Welcome to Clef.md                         6
Effective Meetings.md                      5
Objectives and Key Results.md              5
Continuing Education.md                    4
One on Ones.md                             4
Other Protected Absences.md                4
Clef Values.md                             4
Salary and Equity Compensation.md          3
Product Manifesto.md                       2
Complaint Policy.md                        2
Sharing Files.md                           2
Hack Weeks.md                              1
Equal Opportunity Employment.md            1
Drug and Alcohol Policy.md                 1
Direct Reports.md                          1
At-Will Employment.md                      1
Budgeting.md                               1
Code of Conduct in the Community.md        1
H

## 30. Check chunks by category

In [38]:
chunks_per_category = (
    chunks_df
    .groupby("category")
    .size()
    .sort_values(
        ascending=False
    )
)

chunks_per_category

category
Onboarding Documents    28
Employment Policies     27
General                 17
Benefits and Perks      14
Operations Documents     9
Hiring Documents         1
dtype: int64

## 31. Find very small chunks

Very small chunks aren't automatically wrong, but we should inspect them.

In [39]:
small_chunks = chunks_df[
    chunks_df["word_count"] < 30
].sort_values(
    "word_count"
)

print(
    "Chunks below 30 words:",
    len(small_chunks)
)

small_chunks[
    [
        "document",
        "section_title",
        "word_count",
        "content"
    ]
].head(20)

Chunks below 30 words: 4


,document,section_title,word_count,content
76,Effective Meetings.md,Effective meetings and group work,24,To increase flexibility in how and where emplo...
77,Effective Meetings.md,Meetings start on time,27,"If you're leading a meeting, it's your respons..."
10,Other Protected Absences.md,Other Leaves of Absence,28,Clef will grant leaves of absence for other ac...
50,Communication and Transparency.md,Calendar Updates,29,"If you're working remotely or from home, you s..."


## 32. Find oversized chunks

In [40]:
oversized_chunks = chunks_df[
    chunks_df["word_count"] > MAX_WORDS
].sort_values(
    "word_count",
    ascending=False
)

print(
    "Chunks above MAX_WORDS:",
    len(oversized_chunks)
)

oversized_chunks[
    [
        "document",
        "section_title",
        "word_count",
        "content"
    ]
]

Chunks above MAX_WORDS: 0


,document,section_title,word_count,content


## 33. Inspect the largest chunks

In [41]:
largest_chunks = (
    chunks_df
    .sort_values(
        "word_count",
        ascending=False
    )
    .head(15)
)

largest_chunks[
    [
        "document",
        "section_title",
        "word_count"
    ]
]

,document,section_title,word_count
11,Referral Bonuses.md,Referral Bonuses,336
68,Product Manifesto.md,Product Manifesto,324
33,Salary and Equity Compensation.md,Salary,314
61,Objectives and Key Results.md,Scoring and Evaluating OKRs,312
20,Complaint Policy.md,Complaint Policy,308
19,Code of Conduct in the Community.md,Code of Conduct,296
36,Working Remotely.md,Approach,291
12,Sabbatical.md,Sabbatical,287
6,New Parent Leave.md,New Parent Leave,266
73,Welcome to Clef.md,"The Clef Team, Hours, and Office",264


In [42]:
print(
    largest_chunks.iloc[0]["content"]
)

Referrals from people who already work at Clef are one of the best signals we can get about whether or not a candidate will be successful at Clef. Clef employees understand Clef’s needs and what it’s like to work at Clef, and their relationship with the candidate means they know a lot more than we can learn during an interview process.

In order to get good referrals, everyone needs to know which positions are open, know how to start the recruiting process, have time to search their network, and feel aligned with the company incentives. Open positions should be posted on getclef.com/about and also in the #hiring channel so that everyone sees who we’re trying to hire. We’ll host a company lunch with food on the second Tuesday of every month where we talk about which positions are open and go through everyone’s online networks looking for and thinking about candidates. Employees will do initial outreach to interesting connection, which will give us a good space to talk about how we start

## 34. Check for empty chunks

In [43]:
empty_chunks = chunks_df[
    chunks_df["content"]
    .fillna("")
    .str.strip()
    == ""
]

print(
    "Empty chunks:",
    len(empty_chunks)
)

Empty chunks: 0


## 35. Check for duplicate chunks

This is important for RAG.

In [44]:
duplicate_chunks = (
    chunks_df[
        chunks_df.duplicated(
            subset=["content"],
            keep=False
        )
    ]
    .sort_values("content")
)

print(
    "Duplicate chunk records:",
    len(duplicate_chunks)
)

Duplicate chunk records: 0


## 36. Check section paths

This validates that our structural parsing actually preserved the hierarchy.

In [45]:
section_distribution = (
    chunks_df[
        [
            "document",
            "section_path"
        ]
    ]
    .assign(
        section_path=lambda df: df["section_path"].apply(tuple)
    )
    .drop_duplicates()
)

section_distribution["section_path"] = (
    section_distribution["section_path"].apply(list)
)

section_distribution.head(30)

,document,section_path
0,Continuing Education.md,[Continuing Education]
1,Continuing Education.md,"[Continuing Education, Learning Budget]"
2,Continuing Education.md,"[Continuing Education, Mentorship]"
3,Continuing Education.md,"[Continuing Education, Speaker Support]"
4,Healthcare and Disability Insurance.md,[Healthcare and Disability Insurance]
5,Holiday List.md,[Clef Observed Holiday List]
6,New Parent Leave.md,[New Parent Leave]
7,Other Protected Absences.md,"[Other Protected Absences, Pregnancy Disabilit..."
8,Other Protected Absences.md,"[Other Protected Absences, Bereavement Leave]"
9,Other Protected Absences.md,"[Other Protected Absences, Jury Duty or Witnes..."


In [46]:
from collections import defaultdict

# Build a tree from section paths
tree = defaultdict(dict)

for section_path in chunks_df["section_path"].dropna():

    if not isinstance(section_path, list):
        continue

    current = tree

    for section in section_path:
        if section not in current:
            current[section] = {}

        current = current[section]


def print_tree(tree, level=0):
    for section, children in tree.items():

        indent = "    " * level

        if level == 0:
            print(f"{indent}{section}")
        else:
            print(f"{indent}→ {section}")

        print_tree(children, level + 1)


print_tree(tree)

Continuing Education
    → Learning Budget
    → Mentorship
    → Speaker Support
Healthcare and Disability Insurance
Clef Observed Holiday List
New Parent Leave
Other Protected Absences
    → Pregnancy Disability Leave
    → Bereavement Leave
    → Jury Duty or Witness Summons
    → Other Leaves of Absence
Referral Bonuses
Sabbatical
Vacation and Sick Leave
Clef Core Values
    → Be better today than yesterday.
    → Treat others the way they'd like to be treated.
    → Fight the default of exclusion.
    → We succeed together when we trust each other.
At-Will Employment Policy
Code of Conduct
Complaint Policy
Drug and Alcohol Policy
Employee Privacy
    → Workspace Privacy
    → Email and Internet Privacy
    → Email Is Not Private
    → Use of the Email System for Personal Email
    → All Conduct Rules Apply to Email
    → Professional Tone and Content
    → Email Security
    → Internet Use Is Not Private
Equal Opportunity Employment
Salary and Equity Compensation
    → Salary
Work

37. Inspect random chunks manually

This is one of the most useful validation steps.

In [47]:
sample_chunks = chunks_df.sample(
    min(10, len(chunks_df)),
    random_state=42
)

for _, row in sample_chunks.iterrows():

    print("=" * 80)

    print(
        f"Document: {row['document']}"
    )

    print(
        f"Section: {' → '.join(row['section_path'])}"
    )

    print(
        f"Words: {row['word_count']}"
    )

    print()

    print(row["content"])

    print()

Document: Effective Meetings.md
Section: Effective meetings and group work → Meeting Ettiquette → Prerequisites for successful meetings
Words: 62

The following things are pre-requisites for successful group work:

* A fast, reliable internet connection. Remote employees should make all efforts to mitigate video call lag.
* A quiet place to take meetings

If you're planning to work with another person on the team (i.e. for a meeting during meeting hours or to pair program), you should make sure these are available.

Document: Effective Meetings.md
Section: Effective meetings and group work → Meeting Ettiquette → Meetings start on time
Words: 27

If you're leading a meeting, it's your responsibility to start the meeting on time. If you're attending a meeting, you are responsible for showing up on time.

Document: Welcome to Clef.md
Section: Welcome to Clef → The Clef Team, Hours, and Office
Words: 264

For now, Clef operates as a single team where everyone is working towards company-lev

38. Save the chunks as JSONL

JSONL is particularly useful later for embedding pipelines.

In [48]:
jsonl_path = CHUNK_DIR / "chunks.jsonl"

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as f:

    for record in all_chunks:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

print(
    "Saved:",
    jsonl_path.resolve()
)

Saved: E:\Projects\Speech AI\Data\chunked_data\chunks.jsonl


39. Save a CSV for easy inspection

In [49]:
csv_path = CHUNK_DIR / "chunks.csv"

chunks_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    csv_path.resolve()
)

Saved: E:\Projects\Speech AI\Data\chunked_data\chunks.csv


40. Create a summary report

This gives you a quick overview of the dataset.

In [50]:
summary = {
    "documents": len(documents),

    "total_chunks": len(chunks_df),

    "average_words_per_chunk": round(
        chunks_df["word_count"].mean(),
        2
    ),

    "median_words_per_chunk": round(
        chunks_df["word_count"].median(),
        2
    ),

    "minimum_words": int(
        chunks_df["word_count"].min()
    ),

    "maximum_words": int(
        chunks_df["word_count"].max()
    ),

    "chunks_above_max": len(
        chunks_df[
            chunks_df["word_count"] > MAX_WORDS
        ]
    ),

    "chunks_below_30_words": len(
        chunks_df[
            chunks_df["word_count"] < 30
        ]
    ),

    "empty_chunks": len(
        empty_chunks
    ),

    "duplicate_chunk_records": len(
        duplicate_chunks
    )
}

summary

{'documents': 30,
 'total_chunks': 96,
 'average_words_per_chunk': np.float64(121.05),
 'median_words_per_chunk': np.float64(97.5),
 'minimum_words': 24,
 'maximum_words': 336,
 'chunks_above_max': 0,
 'chunks_below_30_words': 4,
 'empty_chunks': 0,
 'duplicate_chunk_records': 0}

41. Save the summary

In [51]:
summary_path = CHUNK_DIR / "chunk_summary.json"

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print(
    "Saved:",
    summary_path.resolve()
)

Saved: E:\Projects\Speech AI\Data\chunked_data\chunk_summary.json


In [52]:
HEADING_PATTERN = re.compile(r"^\s*(#{1,6})\s*(.+?)\s*$")

In [58]:
!pip install BeautifulSoup4

In [63]:
from bs4 import BeautifulSoup

def convert_html_tables(text):
    soup = BeautifulSoup(text, "html.parser")

    for table in soup.find_all("table"):
        rows = []

        for row in table.find_all("tr"):
            cells = row.find_all(["th", "td"])

            values = [
                cell.get_text(" ", strip=True)
                for cell in cells
            ]

            if values:
                rows.append(" — ".join(values))

        table.replace_with("\n" + "\n".join(rows) + "\n")

    return str(soup)

In [64]:
documents = []

for file_path in markdown_files:

    content = file_path.read_text(
        encoding="utf-8"
    )

    content = convert_html_tables(content)

    documents.append({
        "file_path": str(
            file_path.relative_to(CLEANED_DIR)
        ),
        "file_name": file_path.name,
        "category": get_document_category(file_path),
        "content": content
    })

print("Documents loaded:", len(documents))

Documents loaded: 30


In [65]:
print(type(chunks))
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i}")
    print("Type:", type(chunk))
    print("Value:", chunk if isinstance(chunk, str) else chunk.keys())

<class 'list'>
Number of chunks: 96

Chunk 0
Type: <class 'dict'>
Value: dict_keys(['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content'])

Chunk 1
Type: <class 'dict'>
Value: dict_keys(['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content'])

Chunk 2
Type: <class 'dict'>
Value: dict_keys(['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content'])

Chunk 3
Type: <class 'dict'>
Value: dict_keys(['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content'])

Chunk 4
Type: <class 'dict'>
Value: dict_keys(['chunk_id', 'document', 'file_path', 'category', 'section_path', 'section_title', 'chunk_index', 'word_count', 'character_count', 'content'])


In [66]:
all_chunks = []
global_chunk_id = 1

for doc in documents:

    sections = parse_markdown_structure(
        doc["content"]
    )

    for section in sections:

        if not section["content"].strip():
            continue

        chunks = create_content_chunks(
            section["content"],
            target_words=TARGET_WORDS,
            max_words=MAX_WORDS
        )

        for chunk_index, chunk in enumerate(
            chunks, start=1
        ):

            all_chunks.append({
                "chunk_id": f"chunk_{global_chunk_id:06d}",
                "document": doc["file_name"],
                "file_path": doc["file_path"],
                "category": doc["category"],
                "section_path": section["section_path"],
                "section_title": (
                    section["section_path"][-1]
                    if section["section_path"]
                    else doc["file_name"]
                ),
                "chunk_index": chunk_index,
                "word_count": len(chunk.split()),
                "character_count": len(chunk),
                "content": chunk
            })

            global_chunk_id += 1

print("Total chunks:", len(all_chunks))

Total chunks: 96


In [60]:
chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
print("Unique IDs:", chunks_df["chunk_id"].nunique())
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())

print(
    "Empty chunks:",
    chunks_df["content"].fillna("").str.strip().eq("").sum()
)

print(
    "HTML table chunks:",
    chunks_df["content"]
    .str.contains(
        r"<table|</table>|<tr|</tr>|<td|</td>",
        case=False,
        regex=True,
        na=False
    ).sum()
)

print(
    "Chunks > MAX_WORDS:",
    (chunks_df["word_count"] > MAX_WORDS).sum()
)

print(
    "Average words:",
    round(chunks_df["word_count"].mean(), 2)
)

print(
    "Median words:",
    round(chunks_df["word_count"].median(), 2)
)

Total chunks: 96
Unique IDs: 96
Duplicate IDs: 0
Empty chunks: 0
HTML table chunks: 0
Chunks > MAX_WORDS: 0
Average words: 120.88
Median words: 96.0


In [67]:
jsonl_path = CHUNK_DIR / "chunks.jsonl"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in all_chunks:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("Saved:", jsonl_path.resolve())

Saved: E:\Projects\Speech AI\Data\chunked_data\chunks.jsonl
